# 10 · Postprocessing & evaluation

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=10-postprocessing.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/10-postprocessing.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>
:::


A solution is only as good as what we can *get out of it*: error norms,
convergence rates, integrals, fluxes, point values and pictures. We use the
**method of manufactured solutions**: *choose* the exact answer
$u(x,y)=\sin(\pi x)\sin(\pi y)$ and let NGSolve **differentiate it for us** with
`.Diff` — the gradient $\nabla u$ and the matching right-hand side $f=-\Delta u$
come straight out, no calculus by hand. Then we can measure the error.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from math import pi

uex     = sin(pi*x)*sin(pi*y)
graduex = CF((uex.Diff(x), uex.Diff(y)))                       # ∇u, by symbolic .Diff
rhs     = -(uex.Diff(x).Diff(x) + uex.Diff(y).Diff(y))        # f = -Δu  (= 2π² u here)

def solve(maxh, order=3):
    mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))
    fes = H1(mesh, order=order, dirichlet=".*")
    u, v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
    f = LinearForm(rhs*v*dx).Assemble()
    gfu = GridFunction(fes)
    gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
    return mesh, gfu

mesh, gfu = solve(0.1)
Draw(gfu, mesh)

## 1. Error norms

`Integrate` of a `CoefficientFunction` gives any integral functional — here the
$L^2$ and $H^1$-seminorm errors against the exact solution.

In [ ]:
print("L2 error :", sqrt(Integrate((gfu - uex)**2, mesh)))
print("H1 error :", sqrt(Integrate((grad(gfu) - graduex)**2, mesh)))

## 2. Convergence

Halving `maxh` should reduce the $L^2$ error by $\approx 2^{\,k+1}$ for order
$k$. Let us check.

In [ ]:
print(f"{'maxh':>6} {'L2 error':>12} {'rate':>6}")
prev = None
for h in [0.2, 0.1, 0.05, 0.025]:
    m, g = solve(h)
    err = sqrt(Integrate((g - uex)**2, m))
    rate = "" if prev is None else f"{log(prev/err)/log(2):.2f}"
    print(f"{h:>6} {err:>12.2e} {rate:>6}")
    prev = err

## 3. Functionals: mean value and flux

A volume average is just an integral. A **boundary flux**
$\int_\Gamma \nabla u\cdot n\,ds$ needs care: the bare `grad(gfu)*n` on a
boundary evaluates the *tangential* trace and silently returns 0 — wrap it in
`BoundaryFromVolumeCF` to take the gradient from the volume side.

In [ ]:
n = specialcf.normal(2)
print("mean value           :", Integrate(gfu, mesh))
print("flux through 'bottom':", Integrate(BoundaryFromVolumeCF(grad(gfu)*n),
                                          mesh.Boundaries("bottom")))
print("(naive grad*n, wrong):", Integrate(grad(gfu)*n, mesh.Boundaries("bottom")))

## 4. Point evaluation

Evaluate the solution (or any expression) at a point via `mesh(x, y)`.

In [ ]:
print("u(0.5, 0.5) =", gfu(mesh(0.5, 0.5)), "  (exact: 1)")

## 5. Pictures

Draw the **gradient** as a vector field, and warp the solution into a 3D
surface with `deformation=True`.

In [ ]:
Draw(grad(gfu), mesh, vectors={"grid_size": 24})

In [ ]:
Draw(gfu, mesh, deformation=True)

## 6. VTK export

For ParaView and friends, write a `.vtu` file (with `subdivision` to resolve
high-order detail).

In [ ]:
VTKOutput(mesh, coefs=[gfu, grad(gfu)], names=["u", "grad_u"],
          filename="poisson_post", subdivision=2).Do()
print("wrote poisson_post.vtu")

## 7. Optional — a 3D render with PyVista

`webgui` is perfect inline, but sometimes you want a full 3D toolkit.
[PyVista](https://pyvista.org) can read the `.vtu` we just wrote and warp the
solution into a surface. This section is **entirely optional**: if PyVista is
not installed it simply skips itself, so the notebook still runs everywhere.

In [ ]:
import sys, os, subprocess, pathlib

# Render in a SUBPROCESS: headless OpenGL can hard-crash (segfault) the Python
# process, which a try/except cannot catch — isolating it keeps this kernel (and
# the site build) alive whether or not the GL stack works here. We force Mesa's
# software renderer (llvmpipe) so it works on a headless CI machine too.
_render = r'''
import pyvista as pv
pv.OFF_SCREEN = True
try: pv.start_xvfb()
except Exception: pass
g = pv.read("poisson_post.vtu").warp_by_scalar("u", factor=0.4)
p = pv.Plotter(off_screen=True, window_size=(560, 420))
p.add_mesh(g, scalars="u", cmap="viridis", show_edges=True)
p.camera_position = "xz"; p.camera.elevation = -55
p.screenshot("poisson_pyvista.png")
'''
pathlib.Path("poisson_pyvista.png").unlink(missing_ok=True)
try:
    subprocess.run([sys.executable, "-c", _render], timeout=180, capture_output=True,
                   env=dict(os.environ, LIBGL_ALWAYS_SOFTWARE="1", GALLIUM_DRIVER="llvmpipe"))
except Exception:
    pass                                            # no subprocess (e.g. JupyterLite)

if pathlib.Path("poisson_pyvista.png").exists():
    from IPython.display import Image, display
    display(Image("poisson_pyvista.png"))           # generated at build time → shown on the website
else:
    print("PyVista 3D render unavailable in this environment — skipping (optional).")

:::{dropdown} 📚 Further reading
:class: further-reading

- i-tutorial [1.2 CoefficientFunctions](https://docu.ngsolve.org/latest/i-tutorials/unit-1.2-coefficient/coefficientfunction.html).
- **Error analysis & a posteriori estimates** — Schöberl's iFEM:
  [finite element error analysis](https://jschoeberl.github.io/iFEM/FEM/erroranalysis.html) and
  [a posteriori error estimates](https://jschoeberl.github.io/iFEM/aposteriori/aposteriori.html).
- **Adaptive refinement, hands-on** — the ngs24 tutorial
  [error estimation & adaptivity](https://docu.ngsolve.org/ngs24/tutorials/06_adaptivity.html).
:::

:::{dropdown} 🧠 Quiz — what convergence rate did you observe, and why?
:class: quiz
For order $k$ elements the $L^2$ error of a smooth solution decreases like
$h^{\,k+1}$ — so for $k=3$ you should see a rate close to **4** (each halving of
`maxh` cuts the error by ~16). If you ever see a *lower* rate, suspect a
non-smooth solution (re-entrant corners!), an under-resolved geometry
(`mesh.Curve`), or a bug in the right-hand side.
:::

That completes the core workflow. Next we set the cup in **motion**: a
time-dependent heat problem.

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "11-transient-cooling", "11 · Cooling down in time ☕"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))